# BattMo Workshop

The workshop consists of the following parts:
- **Hello BattMo**: a short introduction to BattMo.jl
- **P4D modelling**: explore how to simulate a 3D geometry in BattMo.jl
- **Thermal modelling**: explore the thermal model in BattMo.jl and use the by the dS-Toolbox computed entropy coefficient to improve our thermal results

##### Import packages

In [ ]:
using BattMo, Plots, Unitful, Jutul, CSV, DataFrames, Statistics
include("./input_files/function_parameters_xu_2015.jl")

include("./scripts/plotting_functions.jl")

gr()

## Hello BattMo

Welcome to this hands-on tutorial where we’ll explore the basics of BattMo.jl — a powerful Julia package for simulating lithium-ion battery cells using physics-based models like the Doyle-Fuller-Newman (DFN) model. 

By the end of this part of the tutorial, you’ll:

- Understand some basic features of BattMo.jl
- Run your first battery simulation
- Explore and visualize the output
- Learn how to tweak key parameters for custom behavior

#### Input format

BattMo.jl structures its simulation inputs into two primary categories: Parameters and Settings. This distinction helps users differentiate between the physical characteristics of the battery system and the numerical configurations of the simulation.

**Parameters** represent the controllable variables in real-world experiments. They are further divided into:

- **Cell Parameters**: define the intrinsic properties of the battery cell, such as geometry and material characteristics.
- **Cycling Protocol Parameters**: specify how the cell is operated during a simulation.

**Settings** are used to configure numerical assumptions for solving equations and finding numerical solutions. They are further divided into:

- **Model Settings**: define numerical assumptions related to the battery model, such as diffusion methods or simplifications used in the simulation.
- **Simulation Settings**: define numerical assumptions specific to the simulation process, including time-stepping schemes and discretization precision.
- **Solver Settings**: define solver behavior and verbosity.

**BattMo** stores cell parameters, cycling protocols and settings in a user-friendly JSON format to facilitate reuse.

#### Load and modify input files

In [ ]:
include("./input_files/function_parameters_xu_2015.jl")

cell_parameters = load_cell_parameters(; from_file_path = "input_files/cell_parameters.json");
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json");

In [ ]:
print_info(cell_parameters)

To change cell parameters, we can modify the JSON files directly, or we can read them into objects in the script and modify them as Dictionaries. A loaded cell parameter set is a Dictionary-like object which come with additional handy functions like the `print_info` we just used.

Parameters that take single numerical values (e.g. real, integers, booleans) can be directly modified.

In [ ]:
cell_parameters["PositiveElectrode"]["Coating"]["Thickness"] = 92.0e-6

Some parameters are described as functions or arrays, since the parameter value depends on other variables. For instance the Open Circuit Potentials of the Active Materials depend on the lithium stoichiometry and temperature. When we're unsure about the type or meaning of a parameter, we can print information on invidual parameters as well. For some parameters, that require more explanation, a link to the documentation is provided. Visit the documentation of the OpenCircuitPotential parameter to find more information on how to implement you own user defined functional parameters.

In [ ]:
parameter_name = "OpenCircuitPotential"

print_info(parameter_name, view = "CellParameters")

The cycling protocol parameters and the settings (model settings, simulation settings, solver settings) can be loaded, viewed and altered in the same way as the cell parameters. Let's load the CC discharge protocol that we have in our input files. The settings we'll go into later.

In [ ]:
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json");
print_info(cycling_protocol)

#### Run a simulation

In this part we'll show how you can run a simple P2D simulation. Let's start by loading the Xu 2015 cell parameter set and a constant current discharge cycling protocol.

In [ ]:
cell_parameters = load_cell_parameters(; from_file_path = "input_files/cell_parameters.json");
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json");

Next, we select the default Lithium-Ion Battery model. A model can be thought as a mathematical implementation of the electrochemical and transport phenomena occuring in a real battery cell. The implementation consist of a system of partial differential equations and their corresponding parameters, constants and boundary conditions. The default Lithium-Ion Battery setup selected below contains the model settings to simulate a basic P2D model, where neither current collectors nor SEI growth are considered.

In [ ]:
model = LithiumIonBattery();

The LithiumIonBattery constructor validates the model settings on the back ground. If the model setup is valid we can continue and create a Simulation object by passing the model setup, cell parameters and a cycling protocol.

In [ ]:
sim = Simulation(model, cell_parameters, cycling_protocol);

The simulation object is only instantiated when the model provided is valid. We can see that the Simulation object also validates the parameters and settings on the back ground. Each parameter set is validated on whether they are sensible and complete. 

When the Simulation object is valid we can solve the simulation by passing the object to the solve function. As Julia is a compiled language, the first time that we run a simulation it will take some time to compile the functions and structs that it encounters. This makes running a second simulation very fast. See the difference by running the script for a second time.

In [ ]:
output = solve(sim);

We can use some built in functions for quick plotting. The dashboard gives you a quick overview of some important ouput variables. You can choose to have interactive line plots where you can change the time step using a slider or contour plots that show the position and time in one plot.

In [ ]:
plot_dashboard(output; plot_type = "contour")

Within the surface concentration of the positive electrode we can see that only part of the electrode is utilized. Why could this be the case? We always recommend to use our `quick_cell_check` function to quickly get access to some common battery cell KPIs like the cell theoretical capacity, n:p ratio, cell mass, etc. to get a first idea of the parameter set and if any of the equilibrium state quantities seem correct and realistic.

In [ ]:
quick_cell_check(cell_parameters)

From the quick cell check we can see that the n:p ratio of the cell is way below 1.0. As we probably now, in order to make sure a physical cell doesn't short circuit, and uses as much of the positive electrode capacity as possible, the n:p ratio should always be higher than 1.0.

## -Assignment

Adjust the negative electrode coating thickness to improve the electrode balancing (n:p ratio) so that we can run the following simulations during this tutorial with a realistic cell input parameter set. You can use `quick_cell_check` to see if your n:p ratio is improved

Run a simulation with the improved cell parameters and plot the dashboard to see if the change in thickness has influence on the lithium surface concentration of the positive electrode.

#### Handle Output

In BattMo.jl the output variables are divided into three different categories:
- **time series**: all variables that depent on time.
- **states**: all the state variables, which can be dependent on time, axial position and radial position.
- **metrics**: the calculated cell metrics, dependent on the cycle index.

In order to investigate how to retrieve output quantities, let's simulate a couple of constant current constant voltage cycles.

In [ ]:
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cccv.json")

In [ ]:

cycling_protocol["TotalNumberOfCycles"] = 10

model_setup = LithiumIonBattery()

sim = Simulation(model_setup, cell_parameters, cycling_protocol);

output = solve(sim);

plot_dashboard(output, plot_type = "simple")

Let's use the output from the previous CCCV simulation. We can again use the print info function to have a look into which output variables are available.

In [ ]:
print_info(output)

In the overview we can quickly see the aivalable variables and their units. We can also see that the variables have been devided into three categories like explained above: time series, states, and metrics. This has been done to provide some structure to the variables that is intuitive and cleans up the data. We also use these three categories when retrieving data. Let's first retrieve for example time series data like voltage, current and time.

In [ ]:
time_series = output.time_series

t = time_series["Time"]
E = time_series["Voltage"]
I = time_series["Current"];

Let's now retrieve some state variables

In [ ]:
states = output.states

electrolyte_concentration = states["Electrolyte"]["Concentration"]
electrolyte_potential = states["Electrolyte"]["Potential"];

We can also print more information on each individual variable. For example, to look into the difference between the electrode particle concentration and surface concentration.

In [ ]:
print_info("NegativeElectrodeActiveMaterialParticleConcentration")

We can have a better idea of what the variable represents by reading the description and checking the shape of the variable. We can also retrieve some metrics from the output.

In [ ]:
metrics = output.metrics

discharge_capacity = metrics["DischargeCapacity"]
round_trip_efficiency = metrics["RoundTripEfficiency"]
cycle_index = metrics["CycleIndex"];

Let's plot the discharge capacity against its cycle index.

In [ ]:
lw = 2.5
fs = 12

p1 = plot(
    cycle_index, discharge_capacity,
    linewidth = lw,
    ylabel = "Capacity / Ah",
    xlabel = "Cycle number / -",
    title = "Discharge capacity",
    marker = :circle,
    markersize = 4,
    grid = true,
    legend = false,
    ylims = (0.0, 0.3)
)

p2 = plot(
    cycle_index, round_trip_efficiency,
    linewidth = lw,
    ylabel = "Efficiency / %",
    xlabel = "Cycle number / -",
    title = "Round trip efficiency",
    marker = :circle,
    markersize = 4,
    grid = true,
    legend = false
)

plot(
    p1, p2,
    layout = (2,1),
    size = (1200,800),
    tickfontsize = fs,
    guidefontsize = fs,
    titlefontsize = fs + 2
)

## P4D modelling

In order to get an accurate representation of the thermal effects within a cell we need to run the simulation on a 3D geometry instead of a 1D. So, let's have a look into how can simulate the 3D pouch geometry described in our cell parameters.

What kind of models did we use during the previous simulation? The submodels that are activated are specified withint the model. Let's have a look what the model that we instantiated earlier contains.

In [ ]:
model.settings

We can see that it contains a couple of submodel settings. We can see that we ran a P2D simulation, and that's it, not more special than that. From the following print function we can see which submodels are availabe to specify within the model settings.

In [ ]:
print_submodels()

For the following simulations we would like to use the P4D Pouch framework. We can specify this by altering the model. We would also like to model the current collectors in the 3D geometry, so we add those to the settings as well.

In [ ]:
model.settings["ModelFramework"] = "P4D Pouch"
model.settings["CurrentCollectors"] = "Standard"

Instead of modifying the model instance directly, for clearity and repeatability, it's better to setup your own model_settings input file and pass that to the battery model of choice.

In [ ]:
model_settings = load_model_settings(; from_file_path = "input_files/model_settings.json")

model_settings["ModelFramework"] = "P4D Pouch"
model_settings["CurrentCollectors"] = "Standard"

model = LithiumIonBattery(;model_settings);

Let's setup the simulation for a simple CC discharge and visualize our geometry.

In [ ]:
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json");

sim_3d = Simulation(model, cell_parameters, cycling_protocol);

Within BattMo.jl, we have super handy interactive plotting functions where you can view your geometry and play around with results and components. Here a figure of what this looks like:

![3d_plot.png](public/3d_plot.png)

Unfortunetaly, in the Jupyter hub, we're not able to run these interactive plots, so we'll have to do with static plotting... Let's plot our geometry grid.

In [ ]:
grids = sim_3d.grids

plot_grid(grids; include = [
    "NegativeElectrodeActiveMaterial", 
    "NegativeElectrodeCurrentCollector", 
    "Separator",
    "PositiveElectrodeActiveMaterial",
    "PositiveElectrodeCurrentCollector"
    ])

## - Assignment

Let's design our own Pouch geometry! Have a look at the parameter specifications in the documentation to see what parameters you can change: https://battmo.org/BattMo.jl/dev/manuals/user_guide/geometries.

Adjust the cell_parameters object and initiate the Simulation object with the adapted cell parameters to setup the grid for your own geometry design.

Plot your geometry using the `plot_grid` function.

Let's run the simulation for this 3d geometry.

In [ ]:
output_3d = solve(sim_3d);

We can interactively visualize the states on our 3D geometry using our internal plotting tools, but as said before, they won't work on the hub. So for this tutorial, we'll stick to more simple 2D plotting. Let's plot the surface concentration of lithium in the positive electrode.

In [ ]:
time = output_3d.time_series["Time"]
n = length(time)
plot_cell_data_2d(grids, output_3d;
    variable = "SurfaceConcentration",
    include = [
        "PositiveElectrodeActiveMaterial",
    ],
    timestep = n)

## Thermal Modelling

BattMo.jl has two types of thermal models implemented at the moment: decoupled and sequential. Within the decoupled one, the thermal model is simulated only after the electrochemical simulation is fully finished, while the sequential model updates the thermal simulation at every time step after the electrochemical state has ben computed. For this workshop we'll use the sequential model.

In [ ]:
model.settings["ThermalModel"] = "Sequential"

In order to run a thermal simulation, our parameters need to be adapted. Some of the parameters, like for example, the open circuit potential, the diffusion coefficient, and reaction rate constant have been found to be significantly dependent on the temperature. In order to make our OCP, diffucion coefficient, and reaction rate constant input parameters temperature dependent, we need the entropy coeffient computed in the previous tutorial of dS-Toolbox.

Let's have a look at these input values and how they are setup right now.

In [ ]:
cell_parameters["PositiveElectrode"]["ActiveMaterial"]["OpenCircuitPotential"]

Lets first do a simulation without the entropy change

In [ ]:
function entropy_change_function_zero(c, cmax)
    return 0.0
end


const entropy_change_function_zero = entropy_change_function_zero


In [ ]:
cell_parameters_no_entropy = deepcopy(cell_parameters)

cell_parameters_no_entropy["PositiveElectrode"]["ActiveMaterial"]["EntropyChange"] = Dict(
    "FunctionName" => "entropy_change_function_zero",
)

cell_parameters_no_entropy["NegativeElectrode"]["ActiveMaterial"]["EntropyChange"] = Dict(
    "FunctionName" => "entropy_change_function_zero",
)

cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json")

cycling_protocol["InitialTemperature"] = 25 + 273.15
cycling_protocol["AmbientTemperature"] = 30 + 273.15

sim_thermal_no_entropy = Simulation(model, cell_parameters_no_entropy, cycling_protocol);

In [ ]:
output_thermal_no_entropy = solve(sim_thermal_no_entropy);

We can add the entropy coefficient calculated by the dS-Toolbox in a similar way.

Let's load the entropy coefficient data from the dS-Toolbox

In [ ]:
# 1. Read data
path = joinpath(pwd(), "..\\dS-Toolbox\\Entropy_data","BattMo_dUdT.csv")
df = CSV.read(path, DataFrame)

soc = df[:, 1]
entropy_change = df[:, 2];

p = plot(
    soc,
    entropy_change,
    label = "dS-Toolbox",
    linewidth = 2,
    marker = :circle,
    markersize = 4,
    title = "Entropy change vs SOC",
    xlabel = "SOC",
    ylabel = "Entropy change",
    legend = true,
    grid = false
)

plot(
    p,
    size = (1200, 600),
    tickfontsize = fs,
    guidefontsize = fs,
    titlefontsize = fs + 2
)

We can interpolate the data and return the interpolation within a function.

In [ ]:
# 2. Use a Jutul interpolator to create an interpolation object of the time and power data
entropy_intp = Jutul.get_1d_interpolator(soc, entropy_change, cap_endpoints = true)


# 3. Define a function to calculate the entropy change.

function entropy_change_function(c, cmax)
    cm = float(cmax)
    cv = float(c)

    soc = cv / cm

    soc = clamp(soc, 0.0, 1.0)

    return entropy_intp(soc)
end


const entropy_change_function = entropy_change_function


Now we have to let BattMo.jl now that we want it to compute the entropy change using this function. To do this we add the name of the function to the cell parameters input file.

In [ ]:
cell_parameters_entropy = deepcopy(cell_parameters)

cell_parameters_entropy["PositiveElectrode"]["ActiveMaterial"]["EntropyChange"] = Dict(
    "FunctionName" => "entropy_change_function",
)

cell_parameters_entropy["NegativeElectrode"]["ActiveMaterial"]["EntropyChange"] = Dict(
    "FunctionName" => "entropy_change_function",
)

In [ ]:
cycling_protocol = load_cycling_protocol(; from_file_path = "input_files/cc_discharge.json")

cycling_protocol["InitialTemperature"] = 25 + 273.15
cycling_protocol["AmbientTemperature"] = 30 + 273.15

In [ ]:
sim_thermal = Simulation(model, cell_parameters_entropy, cycling_protocol);

Now we can run the thermal simulation

In [ ]:

output_thermal = solve(sim_thermal);

Let's make some nice plots to visualize the results

In [ ]:
print_info(output_thermal)

In [ ]:
states = output_thermal.states
time_series = output_thermal.time_series

# T_max_electrolyte = vec(maximum(states["Electrolyte"]["Temperature"], dims = 2))
time = time_series["Time"]
T_max_ne = vec(maximum(states["NegativeElectrode"]["ActiveMaterial"]["Temperature"], dims = 2))
T_max_pe = vec(maximum(states["PositiveElectrode"]["ActiveMaterial"]["Temperature"], dims = 2))

p = Plots.plot(time, T_max_ne .- 273.15,
    label="T max / degC",
    linewidth=2,
   marker=:circle,
    markersize=4,
    title="Maximum temperature vs Time",
    xlabel="Time",
    ylabel="Temperature / degC",
    legend=true,
    grid=false  # optional, often faster without grid in Jupyter
)

Plots.display(p)

In [ ]:
print_info(output_thermal)

In [ ]:

n = length(time)
plot_cell_data_2d(grids, output_thermal;
    variable = "Temperature",
    unit = "K",
    include = [
        "PositiveElectrodeActiveMaterial",
    ],
    timestep = n
)

Compare the voltage curv with and withouth entropy change

In [ ]:
output_thermal.time_series["Voltage"]


In [ ]:
t1 = output_thermal_no_entropy.time_series["Time"] ./ 3600
v1 = output_thermal_no_entropy.time_series["Voltage"]

t2 = output_thermal.time_series["Time"] ./ 3600
v2 = output_thermal.time_series["Voltage"]

voltage_difference = abs.(v2.-v1);

In [ ]:
p = plot(
    t1,
    voltage_difference,
    linewidth = lw,
    xlabel = "Time / h",
    ylabel = "Voltage / V",
    title = "Voltage difference",
    marker = :circle,
    markersize = 4,
    grid = true,
)

In [ ]:


p = plot(
    t1,
    v1,
    linewidth = lw,
    xlabel = "Time / h",
    ylabel = "Voltage / V",
    title = "Voltage comparison",
    # marker = :circle,
    # markersize = 4,
    grid = true,
    legend = :bottomleft,
    label = "Without entropy change"
)

plot!(
    p,
    t2,
    v2,
    linewidth = lw,
    # marker = :circle,
    # markersize = 4,
    label = "With entropy change"
)

plot!(
    p,
    tickfontsize = fs,
    guidefontsize = fs,
    titlefontsize = fs + 2
)

display(p)

## - Assignment

Let's investigate how hot we can make the battery cell until BattMo.jl's solvers can't keep up anymore. Adjust the initial and ambient temperature and solve the simulation to explore.

Plot the maximum temperature of the positive electrode over time like we did before.